# Query Snowflake-Managed Iceberg Tables from Apache Spark

This notebook connects Spark 4.0 to Snowflake-managed Iceberg tables through Horizon Catalog.
No data is copied — Spark reads the open Iceberg files directly.

**Prerequisites:**
- Conda environment: `conda activate fleet-spark`
- Packages: `pip install pyspark==4.0.0 jupyter python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('config.env')

# Configuration
SNOWFLAKE_ACCOUNT_URL = os.getenv('SNOWFLAKE_ACCOUNT', '').replace('.', '-')
SNOWFLAKE_PAT = os.getenv('SNOWFLAKE_PAT', '')
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE', 'FLEET_DB')
SNOWFLAKE_ROLE = os.getenv('SNOWFLAKE_ROLE', 'ACCOUNTADMIN')

ICEBERG_VERSION = '1.10.1'
SCALA_VERSION = '2.13'

print(f'Account: {SNOWFLAKE_ACCOUNT_URL}')
print(f'Database: {SNOWFLAKE_DATABASE}')

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Fleet Analytics - Iceberg Interop') \
    .master('local[*]') \
    .config('spark.driver.host', '127.0.0.1') \
    .config('spark.driver.bindAddress', '127.0.0.1') \
    .config('spark.jars.packages',
            f'org.apache.iceberg:iceberg-spark-runtime-4.0_{SCALA_VERSION}:{ICEBERG_VERSION},'
            f'org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}') \
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
    .config('spark.sql.catalog.horizon', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.horizon.type', 'rest') \
    .config('spark.sql.catalog.horizon.uri',
            f'https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog') \
    .config('spark.sql.catalog.horizon.credential', SNOWFLAKE_PAT) \
    .config('spark.sql.catalog.horizon.warehouse', SNOWFLAKE_DATABASE) \
    .config('spark.sql.catalog.horizon.scope', f'session:role:{SNOWFLAKE_ROLE}') \
    .config('spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation',
            'vended-credentials') \
    .config('spark.sql.iceberg.vectorization.enabled', 'false') \
    .getOrCreate()

print(f'Spark {spark.version} connected to Horizon Catalog!')

## List Tables

All Iceberg tables (including Dynamic Iceberg Tables) are visible through Horizon Catalog.

In [ ]:
spark.sql('SHOW TABLES IN horizon.RAW').show(truncate=False)
spark.sql('SHOW TABLES IN horizon.ANALYTICS').show(truncate=False)

## Query Structured Data

Read the vehicle registry — standard columnar data.

In [ ]:
spark.sql("""
    SELECT VEHICLE_ID, MAKE, MODEL, YEAR, FLEET_REGION
    FROM horizon.RAW.VEHICLE_REGISTRY
    LIMIT 10
""").show()

## Query VARIANT Data with `variant_get()`

Spark 4.0 supports the Iceberg V3 VARIANT type natively. Use `variant_get()` to extract nested fields from JSON telemetry.

In [ ]:
df = spark.sql("""
    SELECT
        VEHICLE_ID,
        EVENT_TIMESTAMP,
        variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') AS speed_mph,
        variant_get(TELEMETRY_DATA, '$.engine.temperature_f', 'int') AS engine_temp,
        variant_get(TELEMETRY_DATA, '$.location.lat', 'float') AS latitude,
        variant_get(TELEMETRY_DATA, '$.location.lon', 'float') AS longitude
    FROM horizon.RAW.VEHICLE_TELEMETRY_STREAM
    WHERE variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') > 60
    LIMIT 10
""")
df.show()

## Query Dynamic Iceberg Tables

Dynamic Iceberg Tables are just regular Iceberg tables to external engines.

In [ ]:
spark.sql("""
    SELECT *
    FROM horizon.ANALYTICS.DAILY_FLEET_SUMMARY
    ORDER BY summary_date DESC
    LIMIT 10
""").show()

## Aggregate Sensor Readings

Run analytics on the sensor readings table.

In [ ]:
spark.sql("""
    SELECT
        VEHICLE_ID,
        COUNT(*) AS reading_count,
        ROUND(AVG(ENGINE_TEMP_F), 1) AS avg_engine_temp,
        ROUND(AVG(FUEL_CONSUMPTION_GPH), 2) AS avg_fuel_gph
    FROM horizon.RAW.SENSOR_READINGS
    GROUP BY VEHICLE_ID
    ORDER BY avg_fuel_gph DESC
    LIMIT 10
""").show()

In [ ]:
spark.stop()
print('Done! Spark read Snowflake-managed Iceberg tables with zero data movement.')